# Fine-tuning AraT5v2-base — Articles → Ruling (Jinayat, standalone)

**Task:** cited article texts → full ruling headnote (NO situation in input)

**Metrics:** ROUGE-L, BLEU, cosine similarity (sentence-embedding).

**Runtime:** set `Runtime → Change runtime type → T4 GPU` before running.

Upload `train.jsonl`, `val.jsonl`, `test.jsonl` (from `build_dataset.py`) when prompted in the second cell.


## 1. Install dependencies

In [ ]:
!pip -q install "transformers==4.44.2" "datasets==2.21.0" "sentencepiece" "sacrebleu" "rouge-score" "sentence-transformers" "accelerate" -U
print("done")

## 2. Upload the dataset
Upload the three JSONL files produced by `build_dataset.py`.

In [ ]:
from google.colab import files
import os
os.makedirs("data", exist_ok=True)
print("Select train.jsonl, val.jsonl, test.jsonl")
up = files.upload()
for name in up:
    open(f"data/{name}","wb").write(up[name])
print(sorted(os.listdir("data")))

## 3. Load data

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={
    "train":"data/train.jsonl","validation":"data/val.jsonl","test":"data/test.jsonl"})
print(ds)
print("\nExample input:\n", ds["train"][0]["input"][:300])
print("\nExample target:\n", ds["train"][0]["target"][:200])

## 4. Load AraT5v2-base + tokenize

AraT5v2-base (UBC-NLP) is a T5 model pretrained on Arabic. We cap input at 512
tokens and target at 256 (see the length stats from build_dataset — p90 input
~2851 chars ≈ fits 512 tokens for most; long ones truncate).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MODEL_NAME = "UBC-NLP/AraT5v2-base-1024"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

MAX_IN, MAX_OUT = 512, 256
def preprocess(batch):
    x = tokenizer(batch["input"], max_length=MAX_IN, truncation=True)
    y = tokenizer(text_target=batch["target"], max_length=MAX_OUT, truncation=True)
    x["labels"] = y["input_ids"]
    return x
tok = ds.map(preprocess, batched=True, remove_columns=ds["train"].column_names)
print(tok)

## 5. Metrics: ROUGE-L, BLEU, cosine

In [ ]:
import numpy as np, sacrebleu
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer
import torch

# multilingual sentence embedder for cosine similarity
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    dpred = tokenizer.batch_decode(preds, skip_special_tokens=True)
    dref  = tokenizer.batch_decode(labels, skip_special_tokens=True)
    dpred = [p.strip() or " " for p in dpred]; dref = [r.strip() for r in dref]
    # ROUGE-L (mean F)
    rl = np.mean([_rouge.score(r, p)["rougeL"].fmeasure for p, r in zip(dpred, dref)])
    # BLEU (corpus, sacrebleu expects list-of-refs)
    bleu = sacrebleu.corpus_bleu(dpred, [dref]).score
    # cosine similarity (mean over pairs)
    ep = embedder.encode(dpred, convert_to_tensor=True, normalize_embeddings=True)
    er = embedder.encode(dref,  convert_to_tensor=True, normalize_embeddings=True)
    cos = float((ep * er).sum(dim=1).mean())
    return {"rougeL": round(rl,4), "bleu": round(bleu,2), "cosine": round(cos,4)}

## 6. Train

In [ ]:
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer,
                              Seq2SeqTrainingArguments)
collator = DataCollatorForSeq2Seq(tokenizer, model=model)

args = Seq2SeqTrainingArguments(
    output_dir="arat5-jinayat-artonly",
    num_train_epochs=8,
    learning_rate=3e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    predict_with_generate=True,
    generation_max_length=MAX_OUT,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_safetensors=False,
    generation_num_beams=1,
    eval_accumulation_steps=4,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    fp16=True,
    report_to="none",
)
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tok["train"], eval_dataset=tok["validation"],
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)
trainer.train()

## 7. Evaluate on the held-out test set

In [ ]:
test_metrics = trainer.evaluate(tok["test"], metric_key_prefix="test")
print(test_metrics)

## 8. Qualitative examples — look at real generations

In [ ]:
import torch
model.eval()
def generate(text):
    ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_IN).to(model.device)
    out = model.generate(**ids, max_length=MAX_OUT, num_beams=4)
    return tokenizer.decode(out[0], skip_special_tokens=True)

for i in range(5):
    ex = ds["test"][i]
    print("="*80)
    print("INPUT (truncated):", ex["input"][:250], "...")
    print("\nGOLD:", ex["target"][:250])
    print("\nMODEL:", generate(ex["input"])[:250])
    print()

## 9. Save results + model

In [ ]:
import json
with open("test_metrics.json","w",encoding="utf-8") as f:
    json.dump(test_metrics, f, ensure_ascii=False, indent=2)
trainer.save_model("arat5-jinayat-artonly-final")
tokenizer.save_pretrained("arat5-jinayat-artonly-final")
print("saved. Download with the next cell or from the Files panel.")

In [ ]:
# zip + download the fine-tuned model (optional, ~1GB)
!zip -r arat5-jinayat-artonly-final.zip arat5-jinayat-artonly-final >/dev/null
from google.colab import files
files.download("test_metrics.json")
# files.download("arat5-jinayat-artonly-final.zip")  # uncomment if you want the weights